# Dipole Antenna — Radiation, Impedance & Field Visualisation

The **dipole antenna** is the most fundamental radiating element in antenna theory.  
A thin wire of total length $L$ centred on the origin along the $z$-axis, fed at $z=0$, supports a sinusoidal current:

$$I(z) = I_0 \sin\!\bigl(k(L/2 - |z|)\bigr), \qquad |z| \le L/2$$

## Key ideas

| Concept | Description |
|---------|-------------|
| Electrical length $L/\lambda$ | Controls current distribution, pattern shape, impedance |
| Far-field pattern factor | $F(\theta) = \dfrac{\cos(kL\cos\theta/2) - \cos(kL/2)}{\sin\theta}$ |
| Half-wave dipole ($L = \lambda/2$) | Classic reference: $D \approx 2.15$ dBi, $R_{in} \approx 73\,\Omega$ |
| E-plane (contains dipole axis) | Pattern varies with $\theta$; figure-8 for $\lambda/2$ |
| H-plane (perpendicular to axis) | Always omnidirectional (circular) |
| Input impedance | $Z_{in} = R_{in} + jX_{in}$; depends strongly on $L/\lambda$ |

## Far-field expressions

$$E_\theta = j\frac{\eta_0 I_0}{2\pi r}e^{-jkr}\,F(\theta), \qquad H_\phi = \frac{E_\theta}{\eta_0}$$

where $\eta_0 = 120\pi \approx 377\,\Omega$ is the intrinsic impedance of free space.

This notebook provides **interactive visualisations** of geometry, current, radiation patterns, impedance, directivity, and near/far field regions.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import (
    FloatSlider, IntSlider, Dropdown, Checkbox,
    HBox, VBox, Output, Label
)
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------- shared constants & helpers ----------
eta0 = 120 * np.pi          # free-space impedance  ~377 Ohm
c0   = 3e8                  # speed of light

def pattern_factor(theta, L_lam):
    """Dipole far-field pattern factor F(theta).
    
    F(theta) = [cos(kL/2 * cos(theta)) - cos(kL/2)] / sin(theta)
    Handles the sin(theta)->0 singularity with np.where.
    """
    kL2 = np.pi * L_lam   # k*L/2 = (2*pi/lam)*(L/2) = pi * L/lam
    num = np.cos(kL2 * np.cos(theta)) - np.cos(kL2)
    den = np.sin(theta)
    # L'Hopital limit at theta=0 and theta=pi: F -> 0 for most L/lam
    return np.where(np.abs(den) < 1e-10, 0.0, num / den)

def pattern_factor_norm(theta, L_lam):
    """Normalised |F(theta)| with peak = 1."""
    F = np.abs(pattern_factor(theta, L_lam))
    Fmax = F.max()
    return F / Fmax if Fmax > 0 else F

## 1 — Dipole Geometry & Current Distribution

A centre-fed dipole of length $L$ along the $z$-axis carries a sinusoidal standing-wave current:

$$I(z) = I_0 \sin\!\bigl(k(L/2 - |z|)\bigr), \qquad |z| \le L/2$$

Key observations:
- **Short dipole** ($L \ll \lambda$): nearly triangular current, small radiation resistance
- **Half-wave** ($L = \lambda/2$): single half-sine lobe, feed-point current is maximum
- **Full-wave** ($L = \lambda$): current has a null at the feed point ($I(0)=0$), impedance $\to \infty$
- **Longer dipoles** ($L > \lambda$): multiple current lobes with alternating phase

Adjust **L/λ** below and watch the current distribution reshape.

In [ ]:
w1_L = FloatSlider(value=0.5, min=0.1, max=3.0, step=0.05,
                   description='L / λ', continuous_update=False,
                   style={'description_width': '60px'})
out1 = Output()

def draw_current(change=None):
    L_lam = w1_L.value
    L = L_lam              # normalise lambda = 1
    k = 2 * np.pi          # k = 2*pi/lambda, lambda=1
    z = np.linspace(-L/2, L/2, 1000)
    I = np.sin(k * (L/2 - np.abs(z)))
    
    with out1:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6),
                                        gridspec_kw={'width_ratios': [1, 1.3]})
        
        # --- Left: dipole geometry ---
        ax1.plot([0, 0], [-L/2, L/2], 'b-', lw=4, solid_capstyle='round',
                 label=f'Dipole L={L_lam:.2f}λ')
        ax1.plot(0, 0, 'ro', ms=10, zorder=5, label='Feed point')
        ax1.plot(0, L/2, 'k^', ms=8, zorder=5)
        ax1.plot(0, -L/2, 'kv', ms=8, zorder=5)
        ax1.annotate(f'z = +L/2 = {L/2:.2f}λ', (0.02, L/2),
                     fontsize=9, color='gray')
        ax1.annotate(f'z = −L/2 = {-L/2:.2f}λ', (0.02, -L/2),
                     fontsize=9, color='gray')
        ax1.set_xlim(-0.3, 0.5)
        ax1.set_ylim(-L/2 - 0.2, L/2 + 0.2)
        ax1.set_xlabel('x / λ')
        ax1.set_ylabel('z / λ')
        ax1.set_title('Dipole geometry')
        ax1.legend(fontsize=9)
        ax1.set_aspect('equal')
        ax1.grid(alpha=0.3)
        
        # --- Right: current distribution ---
        ax2.plot(I, z, 'b-', lw=2, label='I(z) / I₀')
        ax2.fill_betweenx(z, 0, I, alpha=0.15, color='blue')
        ax2.axhline(0, color='r', ls='--', lw=1, alpha=0.5, label='Feed (z=0)')
        ax2.axvline(0, color='gray', ls='-', lw=0.5)
        
        # mark current nulls
        null_positions = []
        for n in range(1, 20):
            z_null = L/2 - n * 0.5  # sin(k*(L/2-|z|))=0 when k*(L/2-|z|)=n*pi
            if z_null > 0 and z_null < L/2:
                null_positions.append(z_null)
                null_positions.append(-z_null)
            elif z_null == 0:
                null_positions.append(0)
        for zn in null_positions:
            ax2.plot(0, zn, 'rx', ms=8, mew=2)
        if null_positions:
            ax2.plot([], [], 'rx', ms=8, mew=2, label='Current nulls')
        
        # mark current maxima
        max_positions = []
        for n in range(0, 20):
            z_max = L/2 - (2*n+1) * 0.25  # sin = +/-1 when arg = (2n+1)*pi/2
            if abs(z_max) < L/2:
                max_positions.append(z_max)
        for zm in max_positions:
            I_val = np.sin(k * (L/2 - abs(zm)))
            ax2.plot(I_val, zm, 'go', ms=7, zorder=5)
        if max_positions:
            ax2.plot([], [], 'go', ms=7, label='Current maxima')
        
        # feed-point current annotation
        I_feed = np.sin(k * L/2)
        ax2.annotate(f'I(0)/I₀ = {I_feed:.3f}',
                     xy=(I_feed, 0), xytext=(0.5, 0.15),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     fontsize=10, color='red')
        
        ax2.set_xlabel('I(z) / I₀')
        ax2.set_ylabel('z / λ')
        ax2.set_title(f'Current distribution   L = {L_lam:.2f}λ   '
                      f'kL = {2*np.pi*L_lam:.2f} rad')
        ax2.legend(fontsize=8, loc='upper right')
        ax2.grid(alpha=0.3)
        
        plt.tight_layout(); plt.show()

w1_L.observe(draw_current, names='value')
draw_current()
display(VBox([HBox([w1_L]), out1]))

## 2 — E-field & H-field Radiation

In the far field the dominant components are:

$$E_\theta(r,\theta,t) = \frac{\eta_0 I_0}{2\pi r}\,F(\theta)\,\sin(\omega t - kr)$$

$$H_\phi(r,\theta,t) = \frac{E_\theta}{\eta_0} = \frac{I_0}{2\pi r}\,F(\theta)\,\sin(\omega t - kr)$$

Below: 2-D spatial maps of $E_\theta$ and $H_\phi$ in the $xz$-plane (E-plane) for a time snapshot $t/T$.  
The dipole sits at the origin along $z$.

In [3]:
w2_L = FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05,
                   description='L / λ', continuous_update=False,
                   style={'description_width': '60px'})
w2_t = FloatSlider(value=0.0, min=0.0, max=1.0, step=0.02,
                   description='t / T', continuous_update=False,
                   style={'description_width': '60px'})
out2 = Output()

def draw_EH_fields(change=None):
    L_lam = w2_L.value
    t_norm = w2_t.value
    k = 2 * np.pi;  omega = 2 * np.pi  # lambda=1, T=1
    kL2 = np.pi * L_lam
    
    # observation grid in xz-plane
    Ng = 400
    xg = np.linspace(-6, 6, Ng)
    zg = np.linspace(-6, 6, Ng)
    X, Z = np.meshgrid(xg, zg)
    R = np.sqrt(X**2 + Z**2) + 1e-12
    THETA = np.arctan2(np.abs(X), Z)  # angle from z-axis
    # handle both sides: theta in [0,pi]
    THETA = np.arccos(np.clip(Z / R, -1, 1))
    
    # pattern factor
    sin_th = np.sin(THETA)
    F = np.where(np.abs(sin_th) < 1e-10, 0.0,
                 (np.cos(kL2 * np.cos(THETA)) - np.cos(kL2)) / sin_th)
    
    # E_theta field snapshot (real part)
    E_theta = F / R * np.sin(omega * t_norm - k * R)
    H_phi   = E_theta / eta0 * eta0  # same spatial shape, different scale
    # H_phi = F / R * sin(wt - kr) * (1/eta0) — we just show normalised
    
    # mask near the dipole
    mask = R < L_lam * 0.6
    E_theta[mask] = np.nan
    H_phi[mask] = np.nan
    
    with out2:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
        
        vE = np.nanpercentile(np.abs(E_theta), 97)
        if vE < 1e-10: vE = 1.0
        im1 = ax1.pcolormesh(X, Z, E_theta, cmap='RdBu_r', shading='auto',
                             vmin=-vE, vmax=vE, rasterized=True)
        ax1.plot([0, 0], [-L_lam/2, L_lam/2], 'lime', lw=3, solid_capstyle='round')
        ax1.plot(0, 0, 'ro', ms=5, zorder=5)
        ax1.set_xlabel('x / λ');  ax1.set_ylabel('z / λ')
        ax1.set_title(f'E_θ field   L={L_lam:.2f}λ   t={t_norm:.2f}T')
        ax1.set_aspect('equal')
        plt.colorbar(im1, ax=ax1, fraction=0.046, label='E_θ (a.u.)')
        
        # H_phi — same shape but show with different colormap
        H_display = F / R * np.sin(omega * t_norm - k * R)
        H_display[mask] = np.nan
        vH = np.nanpercentile(np.abs(H_display), 97)
        if vH < 1e-10: vH = 1.0
        im2 = ax2.pcolormesh(X, Z, H_display, cmap='PiYG', shading='auto',
                             vmin=-vH, vmax=vH, rasterized=True)
        ax2.plot([0, 0], [-L_lam/2, L_lam/2], 'lime', lw=3, solid_capstyle='round')
        ax2.plot(0, 0, 'ro', ms=5, zorder=5)
        ax2.set_xlabel('x / λ')
        ax2.set_title(f'H_φ field   L={L_lam:.2f}λ   t={t_norm:.2f}T')
        ax2.set_aspect('equal')
        plt.colorbar(im2, ax=ax2, fraction=0.046, label='H_φ (a.u.)')
        
        plt.tight_layout(); plt.show()

for w in [w2_L, w2_t]:
    w.observe(draw_EH_fields, names='value')
draw_EH_fields()
display(VBox([HBox([w2_L, w2_t]), out2]))

## 3 — Radiation Pattern: E-plane

The **E-plane** contains the dipole axis ($z$) and the observation direction.  
The normalised power pattern in this plane is:

$$P_n(\theta) = \left|\frac{\cos(kL\cos\theta/2) - \cos(kL/2)}{\sin\theta}\right|^2 \bigg/ \max$$

Classic shapes:
- $L = \lambda/2$: figure-8 (single lobe each side of broadside)
- $L = \lambda$: narrower main lobe
- $L > \lambda$: side lobes appear; for $L = 1.5\lambda$ lobes tilt off broadside

In [ ]:
w3_L = FloatSlider(value=0.5, min=0.1, max=3.0, step=0.05,
                   description='L / λ', continuous_update=False,
                   style={'description_width': '60px'})
out3 = Output()

theta_fine = np.linspace(0, 2 * np.pi, 3600)
theta_half = np.linspace(1e-6, np.pi - 1e-6, 1800)  # 0 to pi for dB plot

def draw_Eplane(change=None):
    L_lam = w3_L.value
    
    # full polar (0 to 2*pi) — mirror for negative theta
    F_upper = np.abs(pattern_factor(theta_half, L_lam))
    Fmax = F_upper.max() if F_upper.max() > 0 else 1.0
    F_upper_norm = F_upper / Fmax
    
    # for polar plot: theta from 0 to 2*pi
    # upper half: theta 0 to pi (standard spherical)
    # we plot in polar with angle measured from z-axis
    F_polar = pattern_factor_norm(theta_fine, L_lam)
    
    # dB version
    F_dB = 20 * np.log10(F_upper_norm + 1e-10)
    theta_deg = np.degrees(theta_half)
    
    with out3:
        clear_output(wait=True)
        fig = plt.figure(figsize=(14, 6))
        ax1 = fig.add_subplot(121, projection='polar')
        ax2 = fig.add_subplot(122)
        
        # --- Polar plot ---
        ax1.plot(theta_fine, F_polar, lw=2, color='steelblue')
        ax1.fill(theta_fine, F_polar, alpha=0.1, color='steelblue')
        ax1.set_ylim(0, 1.1)
        ax1.set_title(f'E-plane pattern   L = {L_lam:.2f}λ', pad=15)
        ax1.set_theta_zero_location('N')   # z-axis points up
        ax1.set_theta_direction(-1)
        
        # --- Rectangular dB plot ---
        ax2.plot(theta_deg, F_dB, lw=2, color='steelblue')
        ax2.axhline(-3, color='tomato', ls=':', lw=1, label='-3 dB')
        ax2.axvline(90, color='gray', ls='--', lw=0.8, alpha=0.5, label='Broadside')
        ax2.set_xlim(0, 180)
        ax2.set_ylim(-40, 3)
        ax2.set_xlabel('θ (°)')
        ax2.set_ylabel('Normalised pattern (dB)')
        ax2.set_title(f'E-plane pattern (dB)   L = {L_lam:.2f}λ')
        ax2.legend(fontsize=8)
        ax2.grid(alpha=0.3)
        
        plt.tight_layout(); plt.show()

w3_L.observe(draw_Eplane, names='value')
draw_Eplane()
display(VBox([HBox([w3_L]), out3]))

/var/folders/sn/ms08v0xn03sdzyg1814wfjzc0000gn/T/ipykernel_21426/1500306420.py:28: RuntimeWarning: invalid value encountered in divide
  return np.where(np.abs(den) < 1e-10, 0.0, num / den)


## 4 — Radiation Pattern: H-plane

The **H-plane** is perpendicular to the dipole axis.  
Because the dipole has azimuthal symmetry about $z$, the H-plane pattern is a **circle** (omnidirectional).

$$F_H(\phi) = F(\theta = 90°) = \text{const}$$

Below: both E-plane and H-plane on the same polar plot for comparison.  
The H-plane is always a circle whose radius equals the E-plane pattern evaluated at $\theta = 90°$.

In [ ]:
w4_L = FloatSlider(value=0.5, min=0.1, max=3.0, step=0.05,
                   description='L / λ', continuous_update=False,
                   style={'description_width': '60px'})
out4 = Output()

phi_full = np.linspace(0, 2 * np.pi, 3600)

def draw_EH_planes(change=None):
    L_lam = w4_L.value
    
    # E-plane pattern (normalised)
    F_E = pattern_factor_norm(theta_fine, L_lam)
    
    # H-plane: value of pattern at theta=90 degrees (broadside)
    F_at_90 = np.abs(pattern_factor(np.array([np.pi/2]), L_lam))
    F_E_max = np.abs(pattern_factor(theta_fine, L_lam)).max()
    H_level = F_at_90[0] / F_E_max if F_E_max > 0 else 0.0
    F_H = np.full_like(phi_full, H_level)  # constant circle
    
    with out4:
        clear_output(wait=True)
        fig = plt.figure(figsize=(14, 6))
        ax1 = fig.add_subplot(121, projection='polar')
        ax2 = fig.add_subplot(122, projection='polar')
        
        # --- E-plane polar ---
        ax1.plot(theta_fine, F_E, lw=2, color='steelblue', label='E-plane')
        ax1.fill(theta_fine, F_E, alpha=0.1, color='steelblue')
        ax1.set_ylim(0, 1.1)
        ax1.set_title(f'E-plane   L = {L_lam:.2f}λ', pad=15)
        ax1.set_theta_zero_location('N')
        ax1.set_theta_direction(-1)
        ax1.legend(fontsize=8, loc='lower right')
        
        # --- H-plane polar ---
        ax2.plot(phi_full, F_H, lw=2, color='tomato', label='H-plane')
        ax2.fill(phi_full, F_H, alpha=0.1, color='tomato')
        # overlay E-plane for reference
        ax2.plot(theta_fine, F_E, lw=1, color='steelblue', alpha=0.4, label='E-plane (ref)')
        ax2.set_ylim(0, 1.1)
        ax2.set_title(f'H-plane (omnidirectional)   L = {L_lam:.2f}λ', pad=15)
        ax2.legend(fontsize=8, loc='lower right')
        
        plt.tight_layout(); plt.show()

w4_L.observe(draw_EH_planes, names='value')
draw_EH_planes()
display(VBox([HBox([w4_L]), out4]))

/var/folders/sn/ms08v0xn03sdzyg1814wfjzc0000gn/T/ipykernel_21426/1500306420.py:28: RuntimeWarning: invalid value encountered in divide
  return np.where(np.abs(den) < 1e-10, 0.0, num / den)


## 5 — 3D Radiation Pattern

The full 3-D pattern is obtained by rotating the E-plane pattern about the dipole axis ($z$):  
$$U(\theta, \phi) = |F(\theta)|^2$$

In Cartesian coordinates for plotting:
$$x = r\sin\theta\cos\phi, \quad y = r\sin\theta\sin\phi, \quad z = r\cos\theta$$
where $r = |F(\theta)|$ (normalised).

In [ ]:
w5_L = FloatSlider(value=0.5, min=0.1, max=3.0, step=0.05,
                   description='L / λ', continuous_update=False,
                   style={'description_width': '60px'})
out5 = Output()

def draw_3D_pattern(change=None):
    L_lam = w5_L.value
    
    th = np.linspace(1e-3, np.pi - 1e-3, 90)
    ph = np.linspace(0, 2 * np.pi, 120)
    TH, PH = np.meshgrid(th, ph)
    
    F = pattern_factor_norm(TH, L_lam)
    
    # spherical to cartesian
    Xs = F * np.sin(TH) * np.cos(PH)
    Ys = F * np.sin(TH) * np.sin(PH)
    Zs = F * np.cos(TH)
    
    with out5:
        clear_output(wait=True)
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        ax.plot_surface(Xs, Ys, Zs, facecolors=plt.cm.hot(F),
                        rstride=1, cstride=1, alpha=0.85,
                        linewidth=0, antialiased=True)
        
        # draw dipole axis
        ax.plot([0, 0], [0, 0], [-1.2, 1.2], 'b-', lw=2, label='Dipole axis (z)')
        ax.plot([0], [0], [0], 'ro', ms=5)
        
        ax.set_xlabel('x');  ax.set_ylabel('y');  ax.set_zlabel('z')
        ax.set_title(f'3D radiation pattern   L = {L_lam:.2f}λ')
        m = 1.2
        ax.set_xlim(-m, m); ax.set_ylim(-m, m); ax.set_zlim(-m, m)
        ax.set_box_aspect([1, 1, 1])
        
        plt.tight_layout(); plt.show()

w5_L.observe(draw_3D_pattern, names='value')
draw_3D_pattern()
display(VBox([HBox([w5_L]), out5]))

## 6 — Dipole Length Sweep: Pattern Evolution

As $L/\lambda$ increases, the radiation pattern undergoes significant changes:

| $L/\lambda$ | Pattern shape |
|-------------|---------------|
| $\ll 0.5$ | Broad figure-8, low directivity |
| $0.5$ | Classic half-wave figure-8, $D \approx 2.15$ dBi |
| $0.5 \to 1.0$ | Main lobe narrows |
| $1.0$ | Narrower lobe, $R_{in}$ very high |
| $> 1.0$ | Side lobes appear, main lobe may split |
| $1.5$ | Main lobe tilts away from broadside |

Below: a heatmap showing the pattern in dB vs $\theta$ (vertical) and $L/\lambda$ (horizontal).

In [7]:
out6 = Output()

def draw_waterfall(change=None):
    L_values = np.linspace(0.1, 3.0, 300)
    theta_arr = np.linspace(1e-3, np.pi - 1e-3, 360)
    theta_deg = np.degrees(theta_arr)
    
    pattern_map = np.zeros((len(theta_arr), len(L_values)))
    
    for j, L_lam in enumerate(L_values):
        F = np.abs(pattern_factor(theta_arr, L_lam))
        Fmax = F.max()
        if Fmax > 0:
            F = F / Fmax
        pattern_map[:, j] = F
    
    pattern_dB = 20 * np.log10(pattern_map + 1e-10)
    
    with out6:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(14, 7))
        
        im = ax.pcolormesh(L_values, theta_deg, pattern_dB,
                           cmap='inferno', shading='auto',
                           vmin=-30, vmax=0)
        
        # mark half-wave and full-wave
        ax.axvline(0.5, color='cyan', ls='--', lw=1.5, label='L = λ/2')
        ax.axvline(1.0, color='lime', ls='--', lw=1.5, label='L = λ')
        ax.axvline(1.5, color='yellow', ls=':', lw=1.5, label='L = 3λ/2')
        ax.axvline(2.0, color='orange', ls=':', lw=1.5, label='L = 2λ')
        
        ax.set_xlabel('Dipole length  L / λ', fontsize=11)
        ax.set_ylabel('θ (°)', fontsize=11)
        ax.set_title('Pattern evolution vs dipole length')
        ax.legend(fontsize=8, loc='upper right')
        plt.colorbar(im, ax=ax, label='Normalised pattern (dB)')
        
        plt.tight_layout(); plt.show()

draw_waterfall()
display(out6)

Output()

## 7 — Input Impedance vs Length

The input impedance $Z_{in} = R_{in} + jX_{in}$ of a centre-fed dipole depends strongly on its electrical length.

### Radiation resistance
Using the induced EMF method, the radiation resistance can be computed from:
$$R_{rad} = \frac{\eta_0}{2\pi \sin^2(kL/2)}\int_0^\pi \left[\frac{\cos(kL\cos\theta/2) - \cos(kL/2)}{\sin\theta}\right]^2 \sin\theta\, d\theta$$

### Reactance
The reactance depends on both $L/\lambda$ and the wire radius $a$.  
For a thin dipole ($a/\lambda \ll 1$), the reactance is approximated using the Hallén/King model:
$$X_{in} \approx \frac{\eta_0}{2\pi\sin^2(kL/2)}\left[-2S_i(kL) + \cos(kL)\bigl(2S_i(kL) - S_i(2kL)\bigr) + \sin(kL)\bigl(2C_i(kL) - C_i(2kL) - C_i(2ka^2/L)\bigr)\right]$$

where $S_i$ and $C_i$ are the sine and cosine integrals.

Key values:

| $L/\lambda$ | $R_{in}$ (Ω) | $X_{in}$ | Notes |
|-------------|--------------|----------|-------|
| $\ll 0.5$ | $\approx 20(kL)^2$ | Capacitive (large negative) | Short dipole |
| $\approx 0.48$ | $\approx 70$ | $\approx 0$ (resonance) | First resonance |
| $0.5$ | $73.1$ | $+42.5$ | Half-wave dipole |
| $1.0$ | $\sim 200$ — $\infty$ | Very large | Feed at current null |

In [ ]:
w7_a = FloatSlider(value=0.001, min=0.0001, max=0.01, step=0.0001,
                   description='a / λ', continuous_update=False,
                   readout_format='.4f',
                   style={'description_width': '60px'})
out7 = Output()

def Si(x):
    """Sine integral Si(x) = integral_0^x sin(t)/t dt."""
    from scipy.special import sici
    si, ci = sici(x)
    return si

def Ci(x):
    """Cosine integral Ci(x) = -integral_x^inf cos(t)/t dt."""
    from scipy.special import sici
    si, ci = sici(x)
    return ci

def dipole_impedance(L_lam_arr, a_lam):
    """Compute R_in and X_in for centre-fed dipole using induced EMF method.
    
    Based on Balanis, Antenna Theory, Ch. 8.
    """
    R_arr = np.zeros_like(L_lam_arr)
    X_arr = np.zeros_like(L_lam_arr)
    
    for i, L_lam in enumerate(L_lam_arr):
        k = 2 * np.pi     # lambda = 1
        L = L_lam
        kL = k * L
        a = a_lam
        
        # Radiation resistance via numerical integration of pattern
        theta_int = np.linspace(1e-6, np.pi - 1e-6, 5000)
        sin_th = np.sin(theta_int)
        kL2 = kL / 2
        F2 = np.where(np.abs(sin_th) < 1e-10, 0.0,
                      ((np.cos(kL2 * np.cos(theta_int)) - np.cos(kL2)) / sin_th)**2)
        integrand = F2 * sin_th
        integral = np.trapezoid(integrand, theta_int)
        
        # R_rad = eta/(2*pi) * integral / sin^2(kL/2)
        sin2_kL2 = np.sin(kL2)**2
        if sin2_kL2 < 1e-12:
            # near full-wave: impedance -> very high
            R_arr[i] = 1e4
            X_arr[i] = 1e4
            continue
        
        R_in = eta0 / (2 * np.pi) * integral / sin2_kL2
        R_arr[i] = R_in
        
        # Reactance using the induced EMF method (Balanis Eq. 8-60b)
        C1 = 0.5772  # Euler-Mascheroni constant
        ka2_over_L = k * a**2 / L if L > 0 else 1e-20
        
        # terms
        term1 = 2 * Si(kL)
        term2 = np.cos(kL) * (2 * Si(kL) - Si(2 * kL))
        term3 = np.sin(kL) * (2 * Ci(kL) - Ci(2 * kL) - Ci(2 * k * a**2 / L))
        
        X_in = eta0 / (4 * np.pi * sin2_kL2) * (term1 + term2 + term3)
        X_arr[i] = X_in
    
    return R_arr, X_arr

def draw_impedance(change=None):
    a_lam = w7_a.value
    L_arr = np.linspace(0.05, 2.5, 500)
    R_in, X_in = dipole_impedance(L_arr, a_lam)
    
    # clip extreme values for display
    R_clip = np.clip(R_in, 0, 2000)
    X_clip = np.clip(X_in, -2000, 2000)
    
    with out7:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # Resistance
        ax1.plot(L_arr, R_clip, lw=2, color='steelblue')
        ax1.axvline(0.5, color='gray', ls='--', lw=1, alpha=0.5, label='L = λ/2')
        ax1.axvline(1.0, color='gray', ls=':', lw=1, alpha=0.5, label='L = λ')
        ax1.axhline(73.1, color='tomato', ls=':', lw=1, label='73.1 Ω (λ/2)')
        ax1.set_xlabel('L / λ')
        ax1.set_ylabel('R_in (Ω)')
        ax1.set_title(f'Input resistance   a/λ = {a_lam:.4f}')
        ax1.set_ylim(0, 1000)
        ax1.legend(fontsize=8)
        ax1.grid(alpha=0.3)
        
        # Reactance
        ax2.plot(L_arr, X_clip, lw=2, color='tomato')
        ax2.axvline(0.5, color='gray', ls='--', lw=1, alpha=0.5, label='L = λ/2')
        ax2.axvline(1.0, color='gray', ls=':', lw=1, alpha=0.5, label='L = λ')
        ax2.axhline(0, color='k', ls='-', lw=0.5)
        ax2.set_xlabel('L / λ')
        ax2.set_ylabel('X_in (Ω)')
        ax2.set_title(f'Input reactance   a/λ = {a_lam:.4f}')
        ax2.set_ylim(-1000, 1000)
        ax2.legend(fontsize=8)
        ax2.grid(alpha=0.3)
        
        # annotate key points
        idx_half = np.argmin(np.abs(L_arr - 0.5))
        ax1.annotate(f'R = {R_in[idx_half]:.1f} Ω',
                     xy=(0.5, R_in[idx_half]),
                     xytext=(0.7, R_in[idx_half] + 150),
                     arrowprops=dict(arrowstyle='->', color='steelblue'),
                     fontsize=9, color='steelblue')
        ax2.annotate(f'X = {X_in[idx_half]:.1f} Ω',
                     xy=(0.5, X_in[idx_half]),
                     xytext=(0.7, X_in[idx_half] + 200),
                     arrowprops=dict(arrowstyle='->', color='tomato'),
                     fontsize=9, color='tomato')
        
        plt.tight_layout(); plt.show()

w7_a.observe(draw_impedance, names='value')
draw_impedance()
display(VBox([HBox([w7_a]), out7]))

## 8 — Directivity & Gain vs Length

Directivity measures how much an antenna concentrates power in its peak direction compared to an isotropic source:

$$D = \frac{4\pi\, U_{\max}}{P_{rad}} = \frac{4\pi\, |F(\theta)|^2_{\max}}{\int_0^{2\pi}\!\int_0^{\pi} |F(\theta)|^2 \sin\theta\, d\theta\, d\phi}$$

For the dipole (azimuthally symmetric):
$$D = \frac{2\, |F|^2_{\max}}{\int_0^{\pi} |F(\theta)|^2 \sin\theta\, d\theta}$$

Reference values:
- Short dipole ($L \ll \lambda$): $D = 1.5$ (1.76 dBi)
- Half-wave ($L = \lambda/2$): $D = 1.64$ (2.15 dBi)

In [10]:
out8 = Output()

def compute_directivity(L_lam):
    """Compute directivity by numerical integration."""
    theta = np.linspace(1e-6, np.pi - 1e-6, 5000)
    F = np.abs(pattern_factor(theta, L_lam))
    F2 = F**2
    U_max = F2.max()
    P_rad = np.trapezoid(F2 * np.sin(theta), theta)  # integral over theta; *2*pi for phi
    if P_rad < 1e-20:
        return 0.0
    D = 2 * U_max / P_rad  # factor of 2 = 4*pi / (2*pi) from phi integration
    return D

def draw_directivity(change=None):
    L_arr = np.linspace(0.05, 3.0, 300)
    D_arr = np.array([compute_directivity(L) for L in L_arr])
    D_dBi = 10 * np.log10(D_arr + 1e-10)
    
    with out8:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # linear
        ax1.plot(L_arr, D_arr, lw=2, color='steelblue')
        ax1.axvline(0.5, color='gray', ls='--', lw=1, alpha=0.5, label='L = λ/2')
        ax1.axvline(1.0, color='gray', ls=':', lw=1, alpha=0.5, label='L = λ')
        ax1.axhline(1.64, color='tomato', ls=':', lw=1, label='D = 1.64 (λ/2)')
        ax1.axhline(1.5, color='orange', ls=':', lw=1, label='D = 1.5 (short)')
        ax1.set_xlabel('L / λ')
        ax1.set_ylabel('Directivity (linear)')
        ax1.set_title('Directivity vs dipole length')
        ax1.legend(fontsize=8)
        ax1.grid(alpha=0.3)
        ax1.set_ylim(0, 5)
        
        # dBi
        ax2.plot(L_arr, D_dBi, lw=2, color='gold')
        ax2.axvline(0.5, color='gray', ls='--', lw=1, alpha=0.5, label='L = λ/2')
        ax2.axvline(1.0, color='gray', ls=':', lw=1, alpha=0.5, label='L = λ')
        ax2.axhline(2.15, color='tomato', ls=':', lw=1, label='2.15 dBi (λ/2)')
        ax2.axhline(1.76, color='orange', ls=':', lw=1, label='1.76 dBi (short)')
        ax2.set_xlabel('L / λ')
        ax2.set_ylabel('Directivity (dBi)')
        ax2.set_title('Directivity vs dipole length')
        ax2.legend(fontsize=8)
        ax2.grid(alpha=0.3)
        ax2.set_ylim(0, 8)
        
        # annotate half-wave
        idx = np.argmin(np.abs(L_arr - 0.5))
        ax2.annotate(f'{D_dBi[idx]:.2f} dBi',
                     xy=(0.5, D_dBi[idx]),
                     xytext=(0.8, D_dBi[idx] + 1.5),
                     arrowprops=dict(arrowstyle='->', color='gold'),
                     fontsize=10, color='gold', fontweight='bold')
        
        plt.tight_layout(); plt.show()

draw_directivity()
display(out8)

Output()

## 9 — Near-Field vs Far-Field

The space around an antenna is divided into three regions:

| Region | Boundary | Behaviour |
|--------|----------|-----------|
| Reactive near-field | $R < 0.62\sqrt{D^3/\lambda}$ (or $\approx \lambda/2\pi$) | Dominant $1/r^3$ terms; energy stored, not radiated |
| Radiating near-field (Fresnel) | $0.62\sqrt{D^3/\lambda} < R < 2D^2/\lambda$ | Pattern shape changes with distance |
| Far-field (Fraunhofer) | $R > 2D^2/\lambda$ | Pattern stabilises; $1/r$ dependence |

For the near field, we include the full $1/r$, $1/r^2$, and $1/r^3$ terms of the Hertzian dipole model.  
The total electric field of an infinitesimal dipole element $dz'$ at position $z'$:

$$dE_r \propto \cos\theta'\left(\frac{1}{r'^2} + \frac{1}{jkr'^3}\right)e^{-jkr'}$$
$$dE_\theta \propto \sin\theta'\left(\frac{jk}{r'} + \frac{1}{r'^2} + \frac{1}{jkr'^3}\right)e^{-jkr'}$$

Below: field intensity map showing all three regions.

In [ ]:
w9_L = FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05,
                   description='L / λ', continuous_update=False,
                   style={'description_width': '60px'})
w9_range = FloatSlider(value=5.0, min=1.0, max=15.0, step=0.5,
                       description='Range (λ)', continuous_update=False,
                       style={'description_width': '80px'})
out9 = Output()

def draw_nearfar(change=None):
    L_lam = w9_L.value
    rng = w9_range.value
    k = 2 * np.pi
    L = L_lam
    
    # observation grid in xz-plane
    Ng = 300
    xg = np.linspace(-rng, rng, Ng)
    zg = np.linspace(-rng, rng, Ng)
    X, Z = np.meshgrid(xg, zg)
    
    # integrate contributions from current elements along dipole
    N_src = 200
    z_src = np.linspace(-L/2, L/2, N_src)
    dz = z_src[1] - z_src[0]
    I_src = np.sin(k * (L/2 - np.abs(z_src)))  # current distribution
    
    # total E_theta field (sum contributions)
    E_total = np.zeros((Ng, Ng), dtype=complex)
    
    for m in range(N_src):
        # distance from source element to each observation point
        dx = X
        dz_obs = Z - z_src[m]
        r_prime = np.sqrt(dx**2 + dz_obs**2) + 1e-12
        
        # theta' (angle from z-axis at source point)
        sin_th = np.abs(dx) / r_prime
        
        # near-field: include 1/r, 1/r^2, 1/r^3 terms
        # E_theta ~ sin(theta') * [jk/r + 1/r^2 + 1/(jkr^3)] * exp(-jkr)
        kr = k * r_prime
        term_ff  = 1j * k / r_prime                  # 1/r far-field
        term_mid = 1.0 / r_prime**2                   # 1/r^2
        term_nf  = 1.0 / (1j * k * r_prime**3)       # 1/r^3
        
        E_contrib = I_src[m] * sin_th * (term_ff + term_mid + term_nf) * np.exp(-1j * kr) * dz
        E_total += E_contrib
    
    intensity = np.abs(E_total)**2
    intensity /= intensity.max() + 1e-30
    intensity_dB = 10 * np.log10(intensity + 1e-10)
    
    # region boundaries
    D_eff = L_lam  # antenna dimension
    R_reactive = 1.0 / (2 * np.pi)  # lambda / (2*pi), lambda=1
    R_fresnel = 0.62 * np.sqrt(D_eff**3) if D_eff > 0 else 0.1
    R_fraunhofer = 2 * D_eff**2  # 2*D^2/lambda
    
    with out9:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 9))
        
        im = ax.pcolormesh(X, Z, intensity_dB, cmap='hot', shading='auto',
                           vmin=-40, vmax=0, rasterized=True)
        
        # draw dipole
        ax.plot([0, 0], [-L/2, L/2], 'lime', lw=3, solid_capstyle='round')
        ax.plot(0, 0, 'co', ms=4, zorder=5)
        
        # region boundaries as circles
        angles = np.linspace(0, 2 * np.pi, 200)
        if R_reactive < rng:
            ax.plot(R_reactive * np.cos(angles), R_reactive * np.sin(angles),
                    'c--', lw=1.5, label=f'Reactive NF (r = λ/2π = {R_reactive:.2f}λ)')
        if R_fresnel < rng:
            ax.plot(R_fresnel * np.cos(angles), R_fresnel * np.sin(angles),
                    'yellow', ls='--', lw=1.5,
                    label=f'Fresnel (r = {R_fresnel:.2f}λ)')
        if R_fraunhofer < rng:
            ax.plot(R_fraunhofer * np.cos(angles), R_fraunhofer * np.sin(angles),
                    'lime', ls=':', lw=2,
                    label=f'Fraunhofer (r = 2D²/λ = {R_fraunhofer:.2f}λ)')
        
        ax.set_xlabel('x / λ')
        ax.set_ylabel('z / λ')
        ax.set_title(f'Near-field & far-field intensity   L = {L_lam:.2f}λ')
        ax.set_aspect('equal')
        ax.legend(fontsize=8, loc='upper right')
        plt.colorbar(im, ax=ax, fraction=0.046, label='|E|² (dB)')
        
        plt.tight_layout(); plt.show()

for w in [w9_L, w9_range]:
    w.observe(draw_nearfar, names='value')
draw_nearfar()
display(VBox([HBox([w9_L, w9_range]), out9]))